In [15]:
import pandas as pd
import numpy as np
import itertools
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import PassiveAggressiveClassifier,LogisticRegression
from sklearn.naive_bayes import MultinomialNB,GaussianNB,ComplementNB,BernoulliNB,CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

In [17]:
dataset=pd.read_csv("whatsapp_chat_labeled.csv")
dataset

,Date,Time,Name,Chat,Label
0,10/14/26,1:47 PM,Nivetha,Good morning,Normal
1,10/14/26,1:47 PM,Nivetha,How are you?,Normal
2,10/14/26,1:49 PM,Kirthi,Hi,Normal
3,10/14/26,1:50 PM,Kirthi,Good..u?,Normal
4,10/14/26,1:50 PM,Nivetha,Fine😊,Normal
...,...,...,...,...,...
144,10/19/26,12:45 PM,Kirthi,"Sure, come over and I'll give you an empty bowl😂",Normal
145,10/19/26,12:45 PM,Nivetha,"Wow, such hospitality! I deserve better from you.",Flirty
146,10/19/26,12:46 PM,Kirthi,May be you do. I'll consider it😜,Flirty
147,10/19/26,12:46 PM,Nivetha,"I'll remember that. Anyway, I need to get back...",Normal


In [19]:
x=dataset["Chat"]
y=dataset["Label"]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=53)

In [23]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)
print("Unique words:",len(tfidf_vectorizer.get_feature_names_out()))#unique words
print(tfidf_train.toarray())

Unique words: 155
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.59321092 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.66342682]]


In [25]:
param_grid = {'alpha':[0.1,0.01,1.0,5.0]}
grid = GridSearchCV(MultinomialNB(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
The f1_macro value for best parameter {'alpha': 0.01}: 0.5993589743589745
The confusion Matrix:
 [[14 16]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.78      0.47      0.58        30
      Normal       0.50      0.80      0.62        20

    accuracy                           0.60        50
   macro avg       0.64      0.63      0.60        50
weighted avg       0.67      0.60      0.60        50



In [27]:
param_grid = {'C':[0.1,0.01,1.0],'loss':["hinge","squared_hinge"]}
grid = GridSearchCV(PassiveAggressiveClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 6 candidates, totalling 18 fits
The f1_macro value for best parameter {'C': 0.1, 'loss': 'squared_hinge'}: 0.5784825371336813
The confusion Matrix:
 [[13 17]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.76      0.43      0.55        30
      Normal       0.48      0.80      0.60        20

    accuracy                           0.58        50
   macro avg       0.62      0.62      0.58        50
weighted avg       0.65      0.58      0.57        50



In [29]:
param_grid = {'C':[0.1,0.01,1.0],'penalty':['l2'],'solver':['lbfgs','liblinear','newton-cg','newton-cholesky', 'sag', 'saga']}
grid = GridSearchCV(LogisticRegression(random_state=0,max_iter=1000), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
The f1_macro value for best parameter {'C': 1.0, 'penalty': 'l2', 'solver': 'liblinear'}: 0.512987012987013
The confusion Matrix:
 [[10 20]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.71      0.33      0.45        30
      Normal       0.44      0.80      0.57        20

    accuracy                           0.52        50
   macro avg       0.58      0.57      0.51        50
weighted avg       0.61      0.52      0.50        50



In [31]:
param_grid = {'C':[0.1,0.01,1.0],'kernel':['linear','poly','rbf','sigmoid'],'gamma':['scale','auto']}
grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
The f1_macro value for best parameter {'C': 1.0, 'gamma': 'scale', 'kernel': 'linear'}: 0.5993589743589745
The confusion Matrix:
 [[14 16]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.78      0.47      0.58        30
      Normal       0.50      0.80      0.62        20

    accuracy                           0.60        50
   macro avg       0.64      0.63      0.60        50
weighted avg       0.67      0.60      0.60        50



In [33]:
param_grid = {'n_estimators':[10,50,100],'criterion':['gini','entropy','log_loss'],'min_samples_split':[2,3,4],'max_features':['sqrt','log2'],'class_weight':['balanced','balanced_subsample']}
grid = GridSearchCV(RandomForestClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 108 candidates, totalling 324 fits
The f1_macro value for best parameter {'class_weight': 'balanced_subsample', 'criterion': 'log_loss', 'max_features': 'sqrt', 'min_samples_split': 3, 'n_estimators': 10}: 0.512987012987013
The confusion Matrix:
 [[10 20]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.71      0.33      0.45        30
      Normal       0.44      0.80      0.57        20

    accuracy                           0.52        50
   macro avg       0.58      0.57      0.51        50
weighted avg       0.61      0.52      0.50        50



In [35]:
param_grid = {'criterion':['gini','entropy','log_loss'],'splitter':['best','random'],'min_samples_split':[2,3,4],'max_features':['sqrt','log2'],'class_weight':['balanced']}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
The f1_macro value for best parameter {'class_weight': 'balanced', 'criterion': 'gini', 'max_features': 'log2', 'min_samples_split': 3, 'splitter': 'random'}: 0.44191814799503926
The confusion Matrix:
 [[ 7 23]
 [ 4 16]]
The report:
               precision    recall  f1-score   support

      Flirty       0.64      0.23      0.34        30
      Normal       0.41      0.80      0.54        20

    accuracy                           0.46        50
   macro avg       0.52      0.52      0.44        50
weighted avg       0.55      0.46      0.42        50



In [41]:
param_grid = {'n_neighbors':[2,3,4,5],'weights':['uniform','distance'],'algorithm':['auto','ball_tree','kd_tree','brute'],'p':[1,2]}
grid = GridSearchCV(KNeighborsClassifier(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_macro',cv=3) 
grid.fit(tfidf_train, y_train) 
grid_predictions = grid.predict(tfidf_test) 
cm = confusion_matrix(y_test, grid_predictions)
clf_report = classification_report(y_test, grid_predictions)
f1_macro=f1_score(y_test,grid_predictions,average='macro')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("The confusion Matrix:\n",cm)
print("The report:\n",clf_report)

Fitting 3 folds for each of 64 candidates, totalling 192 fits
The f1_macro value for best parameter {'algorithm': 'auto', 'n_neighbors': 2, 'p': 2, 'weights': 'uniform'}: 0.4981934965877158
The confusion Matrix:
 [[11 19]
 [ 6 14]]
The report:
               precision    recall  f1-score   support

      Flirty       0.65      0.37      0.47        30
      Normal       0.42      0.70      0.53        20

    accuracy                           0.50        50
   macro avg       0.54      0.53      0.50        50
weighted avg       0.56      0.50      0.49        50



C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\model_selection\_search.py:1135: UserWarning: One or more of the test scores are non-finite: [       nan 0.42828416 0.4491155  0.4491155         nan 0.38426656
 0.38983845 0.38426656        nan 0.38426656 0.43132638 0.40509789
        nan 0.36108028 0.36937591 0.36108028        nan 0.42828416
 0.4491155  0.4491155         nan 0.38426656 0.38983845 0.38426656
        nan 0.38426656 0.43132638 0.40509789        nan 0.36108028
 0.36937591 0.36108028        nan 0.42828416 0.4491155  0.4491155
        nan 0.38426656 0.38983845 0.38426656        nan 0.38426656
 0.43132638 0.40509789        nan 0.36108028 0.36937591 0.36108028
        nan 0.42828416 0.4491155  0.4491155         nan 0.38426656
 0.38983845 0.38426656        nan 0.38426656 0.43132638 0.40509789
        nan 0.36108028 0.36937591 0.36108028]
  warnings.warn(
